# 14 — DateTime, Timedelta & Period

Temporal data is in every production dataset. This notebook covers datetime creation,
the `dt` accessor, time-based indexing, resampling, timedelta arithmetic, and Period.

In [ ]:
import pandas as pd
import numpy as np

# Sales transactions dataset with rich datetime data
np.random.seed(42)
n = 500

sales = pd.DataFrame({
    'order_id':   range(10001, 10001 + n),
    'customer':   np.random.randint(1, 200, n),
    'amount':     np.round(np.random.uniform(100, 50000, n), 2),
    'category':   np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], n),
    'order_date': pd.date_range('2022-01-01', periods=n, freq='8h') + \
                  pd.to_timedelta(np.random.randint(0, 480, n), unit='min'),
    'ship_date':  pd.date_range('2022-01-02', periods=n, freq='8h') + \
                  pd.to_timedelta(np.random.randint(1, 10, n), unit='D'),
})
sales['ship_date'] = pd.to_datetime(sales['ship_date'])

print(sales.dtypes)
sales.head(3)

## 1. Creating Datetime — pd.to_datetime()

In [ ]:
# From strings — multiple formats auto-detected
dates_str = pd.Series(['2023-01-15', '15/03/2023', 'Mar 22, 2023', '2023.12.01'])
dates = pd.to_datetime(dates_str, infer_datetime_format=True)
print(dates)

In [ ]:
# Explicit format — much faster for known formats (skips inference)
dates_us = pd.Series(['01/15/2023', '03/22/2023', '12/01/2023'])
parsed = pd.to_datetime(dates_us, format='%m/%d/%Y')
print(parsed)

# errors='coerce' — invalid → NaT
mixed = pd.Series(['2023-01-15', 'not-a-date', '2023-03-22', None])
print(pd.to_datetime(mixed, errors='coerce'))

In [ ]:
# From dict components
dt_from_parts = pd.to_datetime(pd.DataFrame({
    'year': [2023, 2023, 2024],
    'month': [1, 6, 3],
    'day': [15, 30, 1]
}))
print(dt_from_parts)

## 2. The dt Accessor — Date Components

In [ ]:
dates = sales['order_date']

# Date components
print(f"year:        {dates.dt.year.unique()[:5].tolist()}")
print(f"month:       {dates.dt.month.unique()[:5].tolist()}")
print(f"day:         {dates.dt.day.unique()[:5].tolist()}")
print(f"hour:        {dates.dt.hour.unique()[:5].tolist()}")
print(f"minute:      {dates.dt.minute.unique()[:5].tolist()}")
print(f"day_of_week: {dates.dt.day_of_week.unique()[:5].tolist()}")  # 0=Monday
print(f"day_of_year: {dates.dt.day_of_year.unique()[:5].tolist()}")
print(f"quarter:     {dates.dt.quarter.unique().tolist()}")
print(f"week:        {dates.dt.isocalendar().week.unique()[:5].tolist()}")

In [ ]:
# Human-readable names
print(dates.dt.day_name().value_counts())
print()
print(dates.dt.month_name().value_counts())

In [ ]:
# is_month_start, is_month_end, is_quarter_end, is_year_end
sales['is_weekend'] = sales['order_date'].dt.day_of_week >= 5
print("Weekend orders:")
print(sales['is_weekend'].value_counts())

## 3. Feature Engineering with dt

In [ ]:
sales = sales.assign(
    year        = sales['order_date'].dt.year,
    month       = sales['order_date'].dt.month,
    quarter     = sales['order_date'].dt.quarter,
    hour_bucket = (sales['order_date'].dt.hour // 6) * 6,  # 0/6/12/18
    day_name    = sales['order_date'].dt.day_name(),
    is_weekend  = sales['order_date'].dt.day_of_week >= 5
)

print("Revenue by quarter:")
print(sales.groupby('quarter')['amount'].agg(['sum', 'count']).round(0))

In [ ]:
# Revenue by hour bucket
print(sales.groupby('hour_bucket')['amount'].mean().round(0))

## 4. Timedelta — Duration Arithmetic

In [ ]:
# Shipping duration = ship_date - order_date
sales['fulfillment_days'] = (sales['ship_date'] - sales['order_date']).dt.days

print("Fulfillment time (days):")
print(sales['fulfillment_days'].describe())

In [ ]:
# Creating Timedelta
td = pd.Timedelta('5 days')
td2 = pd.to_timedelta(5, unit='D')
print(f"Timedelta: {td}")

# Timedelta arithmetic
today = pd.Timestamp('2024-01-01')
next_week = today + pd.Timedelta('7 days')
print(f"Next week: {next_week}")

# Date difference in hours
diff_hours = (sales['ship_date'] - sales['order_date']).dt.total_seconds() / 3600
print(f"\nFulfillment hours — mean: {diff_hours.mean():.1f}h")

In [ ]:
# Timedelta components with dt accessor
td_series = pd.to_timedelta([1, 2, 10, 25, 50], unit='D')
print(f"Days:    {td_series.days.tolist()}")
print(f"Seconds: {td_series.seconds.tolist()}")
print(f"Total hours: {(td_series.total_seconds() / 3600).tolist()}")

## 5. DatetimeIndex — Time-Based Indexing

In [ ]:
# Set datetime as index for time-series operations
ts = sales.set_index('order_date').sort_index()

# Partial string indexing — select by year/month/day
jan_2022 = ts.loc['2022-01']   # all of January 2022
print(f"January 2022 orders: {len(jan_2022)}")

q1_2022 = ts.loc['2022-01':'2022-03']  # Q1 2022
print(f"Q1 2022 orders: {len(q1_2022)}")

In [ ]:
# between_time — filter by time of day
business_hours = ts.between_time('09:00', '17:00')
print(f"Business hours orders: {len(business_hours)}")

# at_time — exact time
# ts.at_time('09:00')  # all orders placed exactly at 9am

## 6. resample() — Time-Based Aggregation

In [ ]:
# resample() — group by time frequency and aggregate
# 'ME' = Month End, 'W' = Weekly, 'D' = Daily, 'h' = hourly

monthly_revenue = ts['amount'].resample('ME').sum()
print("Monthly Revenue:")
print(monthly_revenue)

In [ ]:
# Weekly order count and revenue
weekly = ts.resample('W').agg(
    orders=('amount', 'count'),
    revenue=('amount', 'sum'),
    avg_order=('amount', 'mean')
).round(0)
print(weekly.head(8))

In [ ]:
# Resample with multiple columns
daily = ts.resample('D').agg({
    'order_id': 'count',
    'amount':   ['sum', 'mean', 'max']
})
daily.columns = ['orders', 'total', 'avg', 'max_order']
print(daily.head(7))

## 7. Date Offsets and Frequencies

In [ ]:
# Common frequency aliases
freqs = {
    'D':  'Calendar day',
    'B':  'Business day',
    'W':  'Weekly',
    'ME': 'Month end',
    'MS': 'Month start',
    'QE': 'Quarter end',
    'YE': 'Year end',
    'h':  'Hourly',
    'min':'Minute',
}
for alias, desc in freqs.items():
    print(f"  '{alias}': {desc}")

In [ ]:
# DateOffset arithmetic
from pandas.tseries.offsets import BDay, MonthEnd, QuarterEnd

date = pd.Timestamp('2023-03-15')
print(f"+ 5 business days: {date + 5 * BDay()}")
print(f"Next month end:    {date + MonthEnd(1)}")
print(f"Next quarter end:  {date + QuarterEnd(1)}")

## 8. Period — Fixed-Frequency Time Intervals

In [ ]:
# Period represents a fixed interval (unlike Timestamp which is an instant)
p = pd.Period('2023-Q1', freq='Q')
print(f"Period: {p}")
print(f"Start: {p.start_time}")
print(f"End:   {p.end_time}")

# Period arithmetic
print(f"Next quarter: {p + 1}")
print(f"2 quarters ago: {p - 2}")

In [ ]:
# PeriodIndex
quarters = pd.period_range('2022Q1', periods=8, freq='Q')
q_revenue = pd.Series(
    np.random.randint(500000, 2000000, 8),
    index=quarters,
    name='revenue'
)
print(q_revenue)

In [ ]:
# Convert between Period and Timestamp
ts_idx = q_revenue.to_timestamp()   # Period → Timestamp (start of period)
back_to_period = ts_idx.to_period('Q')  # Timestamp → Period

print(ts_idx.index)
print(back_to_period.index)

In [ ]:
# dt.to_period() — convert datetime column to period
sales['order_period'] = sales['order_date'].dt.to_period('M')  # monthly periods
monthly_stats = (
    sales
    .groupby('order_period')
    .agg(orders=('order_id', 'count'), revenue=('amount', 'sum'))
    .round(0)
)
print(monthly_stats)

## 9. Timezone Handling

In [ ]:
# Localize a naive datetime to a timezone
naive_ts = pd.Timestamp('2023-01-15 10:30:00')
ist_ts   = naive_ts.tz_localize('Asia/Kolkata')
utc_ts   = ist_ts.tz_convert('UTC')

print(f"Naive IST: {naive_ts}")
print(f"IST (TZ-aware): {ist_ts}")
print(f"UTC:            {utc_ts}")

In [ ]:
# Convert a Series
ts_series = pd.Series(pd.date_range('2023-01-01', periods=5, freq='D'))
ts_ist = ts_series.dt.tz_localize('Asia/Kolkata')
ts_utc = ts_ist.dt.tz_convert('UTC')
print(pd.DataFrame({'IST': ts_ist, 'UTC': ts_utc}))

## 10. Real-World — Customer Cohort by Join Month

In [ ]:
# Cohort analysis — group customers by when they first ordered
# then track their monthly revenue

# Create a simple cohort: first order month
first_order = (
    sales
    .groupby('customer')['order_date']
    .min()
    .dt.to_period('M')
    .rename('cohort')
    .reset_index()
)

cohort_sales = (
    sales
    .merge(first_order, on='customer')
    .assign(order_period=lambda df: df['order_date'].dt.to_period('M'))
    .groupby(['cohort', 'order_period'])['amount']
    .sum()
    .reset_index()
)

print(cohort_sales.head(10))

## 11. Quick Reference — DateTime Cheatsheet

| Operation | Code |
|-----------|------|
| Parse strings | `pd.to_datetime(s, format='%Y-%m-%d')` |
| Extract year | `df['dt'].dt.year` |
| Extract month name | `df['dt'].dt.month_name()` |
| Day of week | `df['dt'].dt.day_of_week` (0=Mon) |
| Is weekend | `df['dt'].dt.day_of_week >= 5` |
| Duration | `(end - start).dt.days` |
| Resample monthly | `df.resample('ME')['col'].sum()` |
| Partial indexing | `df.loc['2023-01']` on DatetimeIndex |
| Add 30 days | `ts + pd.Timedelta('30D')` |
| Month end | `ts + pd.offsets.MonthEnd(1)` |
| To period | `dt.dt.to_period('M')` |
| Localize TZ | `dt.dt.tz_localize('Asia/Kolkata')` |